# IBM Applied Data Science Capstone — Folium Launch-Site Map

This notebook creates an actual interactive Folium map from the IBM course `spacex_launch_geo.csv` data. It shows launch-site markers, individual launch records, landing outcomes, marker clustering, layer controls, pop-ups, and map measurement tools.

For nearest-feature proximity calculations (coastline, railway, highway, city), provide source-verified coordinates in `data/proximity_features.csv`. The template is intentionally blank: this notebook will not invent infrastructure coordinates or claim distances that have not been sourced.

In [1]:
from pathlib import Path
import pandas as pd
import folium
from folium.plugins import MarkerCluster, Fullscreen, MeasureControl, MiniMap
from IPython.display import display, IFrame
from geopy.distance import geodesic
import re, html

## 1. Load and validate the geographical launch dataset

Expected source fields include `Launch Site`, `Lat`/`Latitude`, `Long`/`Longitude`, `class`/`Class`, and optionally `Flight Number`, `Payload Mass (kg)`, and `Serial`.

In [2]:
def locate(filename):
    for p in [Path("data")/filename, Path(filename), Path("../data")/filename]:
        if p.exists(): return p
    raise FileNotFoundError(f"{filename} not found. Place the IBM course CSV in data/.")

def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

path = locate("spacex_launch_geo.csv")
raw = pd.read_csv(path)
lookup = {normalize_name(c): c for c in raw.columns}

def col(*aliases):
    for a in aliases:
        if normalize_name(a) in lookup:
            return lookup[normalize_name(a)]
    return None

site_col = col("Launch Site")
lat_col = col("Lat", "Latitude")
lon_col = col("Long", "Longitude", "Lon")
class_col = col("class", "Class", "Landing Success")
if not all([site_col, lat_col, lon_col, class_col]):
    raise ValueError(f"Required map columns not found. Actual columns: {list(raw.columns)}")

df = raw.rename(columns={site_col:"Launch Site", lat_col:"Latitude", lon_col:"Longitude", class_col:"Class"}).copy()
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
cls = pd.to_numeric(df["Class"], errors="coerce")
if cls.notna().all() and set(cls.unique()).issubset({0,1}):
    df["Class"] = cls.astype(int)
else:
    text = df["Class"].astype(str).str.lower()
    success = text.str.contains("success|landed|true")
    failure = text.str.contains("fail|no attempt|precluded|uncontrolled|controlled|false")
    df["Class"] = pd.NA
    df.loc[success, "Class"] = 1
    df.loc[failure, "Class"] = 0
    df = df.dropna(subset=["Class"])
    df["Class"] = df["Class"].astype(int)

df = df.dropna(subset=["Launch Site", "Latitude", "Longitude", "Class"]).copy()
df["Landing Outcome"] = df["Class"].map({0:"Unsuccessful", 1:"Successful"})
if "Flight Number" not in df.columns:
    df["Flight Number"] = range(1, len(df)+1)
if "Payload Mass (kg)" not in df.columns:
    df["Payload Mass (kg)"] = pd.NA
print("Usable mapped launch records:", len(df))
display(df.head())

Usable mapped launch records: 56


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,Class,Latitude,Longitude
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Unsuccessful,0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Unsuccessful,0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),Unsuccessful,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),Unsuccessful,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),Unsuccessful,0,28.562302,-80.577356


## 2. Summarize the four launch sites

The launch-site marker locations are derived from the coordinate records in the course geographical dataset. The map does not infer landing sites from launch-site coordinates; record markers describe the landing outcome associated with each launch and are positioned at the corresponding launch complex for visualization.

In [3]:
site_summary = df.groupby("Launch Site", as_index=False).agg(
    Latitude=("Latitude", "mean"), Longitude=("Longitude", "mean"),
    Launches=("Flight Number", "count"), Success_Rate=("Class", "mean")
)
site_summary["Success_Rate"] = site_summary["Success_Rate"] * 100
display(site_summary)

,Launch Site,Latitude,Longitude,Launches,Success_Rate
0,CCAFS LC-40,28.562302,-80.577356,26,26.923077
1,CCAFS SLC-40,28.563197,-80.576820,7,42.857143
2,KSC LC-39A,28.573255,-80.646895,13,76.923077
3,VAFB SLC-4E,34.632834,-120.610745,10,40.000000


## 3. Build the interactive Folium map

Map layers include (1) launch-site reference markers, (2) clustered launch records colored by binary first-stage landing outcome, and (3) optional source-verified proximity features. Controls allow the reviewer to toggle layers and measure distances interactively.

In [4]:
center = [df["Latitude"].mean(), df["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=4, control_scale=True, tiles="OpenStreetMap")
Fullscreen(position="topleft").add_to(m)
MiniMap(toggle_display=True).add_to(m)
MeasureControl(primary_length_unit="kilometers", secondary_length_unit="miles").add_to(m)

sites_layer = folium.FeatureGroup(name="Launch-site reference markers", show=True)
records_layer = folium.FeatureGroup(name="Launch records (clustered)", show=True)
cluster = MarkerCluster(name="Launch record cluster").add_to(records_layer)

for _, r in site_summary.iterrows():
    popup = (f"<b>{html.escape(str(r['Launch Site']))}</b><br>Launch records: {int(r['Launches'])}"
             f"<br>Observed landing success rate: {r['Success_Rate']:.1f}%")
    folium.Marker(
        [r["Latitude"], r["Longitude"]],
        tooltip=f"Launch site: {r['Launch Site']}",
        popup=folium.Popup(popup, max_width=320),
        icon=folium.Icon(color="blue", icon="rocket", prefix="fa")
    ).add_to(sites_layer)

for _, r in df.iterrows():
    color = "green" if int(r["Class"]) == 1 else "red"
    payload = r.get("Payload Mass (kg)")
    payload_text = f"<br>Payload: {float(payload):,.0f} kg" if pd.notna(payload) else ""
    popup = (f"<b>Flight:</b> {html.escape(str(r['Flight Number']))}"
             f"<br><b>Launch site:</b> {html.escape(str(r['Launch Site']))}"
             f"<br><b>Landing outcome:</b> {r['Landing Outcome']}" + payload_text)
    folium.CircleMarker(
        [r["Latitude"], r["Longitude"]], radius=6,
        color=color, fill=True, fill_color=color, fill_opacity=0.78,
        tooltip=f"Flight {r['Flight Number']} — {r['Landing Outcome']}",
        popup=folium.Popup(popup, max_width=320)
    ).add_to(cluster)

sites_layer.add_to(m)
records_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
Path("outputs").mkdir(exist_ok=True)
map_path = Path("outputs/spacex_launch_sites_map.html")
m.save(map_path)
print("Interactive map saved to:", map_path.resolve())
display(m)

Interactive map saved to: /home/runner/work/Data-Science-Capstone-Project-Report/Data-Science-Capstone-Project-Report/outputs/spacex_launch_sites_map.html


## 4. Proximity analysis — only with verified source coordinates

Create `data/proximity_features.csv` with columns:
- `Feature Type` — Coastline / Railway / Highway / City
- `Feature Name` — named, identifiable feature
- `Latitude` — source-verified decimal latitude
- `Longitude` — source-verified decimal longitude

Record a citation or source URL for every coordinate in a companion notes file or in the notebook. Do not use the blank template as if it contains measurements.

In [5]:
prox_path = Path("data/proximity_features.csv")
if not prox_path.exists():
    print("Proximity distance table not produced: data/proximity_features.csv is not present.")
    print("Map measurement controls remain available. Add cited feature coordinates to calculate nearest-feature distances.")
else:
    features = pd.read_csv(prox_path)
    required = {"Feature Type", "Feature Name", "Latitude", "Longitude"}
    missing = required - set(features.columns)
    if missing:
        raise ValueError(f"Proximity file missing columns: {sorted(missing)}")
    for c in ["Latitude", "Longitude"]:
        features[c] = pd.to_numeric(features[c], errors="coerce")
    features = features.dropna(subset=["Feature Type", "Feature Name", "Latitude", "Longitude"])
    rows = []
    for _, site in site_summary.iterrows():
        origin = (float(site["Latitude"]), float(site["Longitude"]))
        for feature_type, group in features.groupby("Feature Type"):
            distances = group.apply(
                lambda r: geodesic(origin, (float(r["Latitude"]), float(r["Longitude"]))).kilometers,
                axis=1
            )
            idx = distances.idxmin()
            f = group.loc[idx]
            rows.append({
                "Launch Site": site["Launch Site"],
                "Feature Type": feature_type,
                "Nearest Feature": f["Feature Name"],
                "Distance (km)": float(distances.loc[idx]),
            })
    proximity_results = pd.DataFrame(rows)
    display(proximity_results)
    Path("outputs").mkdir(exist_ok=True)
    proximity_results.to_csv("outputs/proximity_analysis.csv", index=False)

Proximity distance table not produced: data/proximity_features.csv is not present.
Map measurement controls remain available. Add cited feature coordinates to calculate nearest-feature distances.


## 5. Evidence for the final presentation

1. Save a screenshot of the map with all launch-site markers and the layer control visible.
2. Save another screenshot with an expanded cluster or a launch-record popup showing flight number, site, payload (when available), and landing outcome.
3. If verified feature coordinates were supplied, include the generated proximity table and cite the coordinate sources. Otherwise report that proximity was explored using the Measure Control and do not state nearest-feature distances.

The generated HTML file is `outputs/spacex_launch_sites_map.html`. Open it in a browser to demonstrate the actual interactive map. A screenshot of a static coordinate scatter plot is not a substitute for this HTML map.